# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\.venv\Scripts\python.exe
현재 작업 폴더: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\chapter04
프로젝트 루트: C:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study
데이터 폴더: C:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\data\raw
결과 저장 폴더: C:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 1) 질문과 필요한 데이터 선택
### 내가 확인하려는 질문

completed 주문 기준으로 판매 금액은 어떤 카테고리, 상품, 월, 고객에 집중되어 있으며, 이후 어떤 항목을 우선적으로 추가 분석해야 하는가?

### 사용한 파일/컬럼

- orders.csv: 'order_id', 'customer_id', 'order_date', 'order_status'
- order_items.csv: 'order_id', 'product_id', 'quantity', 'unit_price'
- products.csv: 'product_id', 'product_name', 'category', 'price'
- customers.csv: 'customer_id', 'gender', 'age', 'city'

### 결과 관찰

데이터셋별 행, 열 수와 필요한 칼럼의 존재 여부를 확인한 결과, 분석에 필요한 칼럼이 모두 존재했다. 필수 칼럼 누락 시 KeyError가 발생하도록 설정했지만, 오류 없이 실행되었으므로, 이후 단계에 필요한 기본 데이터 구조는 갖춰졌다고 판단했다. 또한, order_items는 764행으로 orders 300행보다 많아, 하나의 주문에 여러 상품이 포함될 수 있는 구조임을 확인할 수 있다.

### 나의 해석과 판단
왜 이 컬럼과 범위를 선택했는지 작성하세요.

완료 주문만 분석하기 위해 orders의 order_status를 사용했고, 월별 분석을 위해 order_date, 고객별 분석을 위해 customer_id를 선택했다.
또한 카테고리와 상품 단위 분석을 위해 products의 product_name, category를 연결했고, 고객의 지역별 특성을 함께 보기 위해 customers의 city를 사용했다. gender, age, price는 고객 세그먼트별 구매 패턴이나 등록 가격과 실제 판매 금액을 비교하는 추가 분석에 활용할 수 있다.
분석 범위는 completed 주문으로 한정했다. 취소·환불 주문을 포함하면 실제 완료된 구매 기준의 판매 패턴을 정확히 보기 어렵기 때문이다.

### 업무·분석적 의미

주문, 주문상세, 상품, 고객 데이터를 함께 사용하면 단순한 전체 금액뿐만 아니라 다음과 같은 의사결정 정보를 만들 수 있다.
- 판매 기여도 높은 카테고리와 상품 파악
- 월별 주문 수와 구매 금액 변화 확인
- 고액 구매 고객 및 고객 지역별 구매 특성 확인

### 한계와 추가 확인 사항

현재 칼럼 존재 여부만 확인했으므로, 이후에 각 키의 고유성 및 병합 결과를 추가로 검증해야 한다. 특히 orders.order_id, products.product_id, customers.customer_id의 중복 여부와 주문 상세의 미매칭 키를 확인할 필요가 있다. 
또한, quantity, unit_price, order_date의 결측치, 비정상값, 날짜 변환 실패 여부도 확인해야 한다. 

## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-15
8,9,송지민,M,69,서울,2025-12-30
54,55,윤영철,M,69,광주,2023-10-11
78,79,이서준,M,69,부산,2023-09-24
136,137,김선영,M,68,울산,2025-02-08
45,46,김서현,M,67,대전,2026-07-24
132,133,김성진,M,67,성남,2026-08-09
123,124,김시우,M,67,대구,2024-12-05
11,12,김정남,F,66,대전,2024-12-11
57,58,김성훈,F,66,성남,2025-08-25


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 2) 필터링·정렬·파생 컬럼
- 적용한 필터 조건: 

-30세 이상인 고객: customers['age'] >= 30
-서울에 거주하는 고객: customers['city'] == '서울'
-서울 또는 부산에 거주하는 고객: customers['city'].isin(['서울', '부산'])

- 정렬 기준: 

-가격 기준 내림차순: products.sort_values('price', ascending=False)
-나이 기준 내림차순: customers.sort_values('age', ascending=False)

- 만든 파생 컬럼: 전체 주문 상세 금액 (line_total)

- `line_total` 계산식: order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

age, city 조건을 이용해 분석 대상을 특정 고객군으로 제한할 수 있고, 내림차순 정렬을 통해 가격이 높은 상품이나 연령이 높은 고객을 우선 확인할 수 있다. 또한, line total을 생성해 주문상세 단위의 수량과 단가를 하나의 금액 지표로 변환했다.

### 나의 해석과 판단
필터 기준이 달라지면 결과가 어떻게 달라질지 작성하세요.

필터 기준이 달라지면 분석 대상과 결과가 함께 달라진다. 예를 들어 서울 거주 고객만 필터링하면 전체 고객의 구매 패턴이 아니라 서울 고객의 구매 패턴을 확인할 수 있다. 서울, 부산 고객으로 범위를 넓히면 두 지역을 포함한 결과가 되므로, 도시별 고객 수와 구매 성향 차이가 결과에 반영될 수 있다.

### 업무·분석적 의미

필터링은 전체 현황 분석 이외에 특정 고객군이나 지역의 구매 특성을 비교하는 데 활용할 수 있다. 정렬은 우선 점검할 상품, 고객을 빠르게 찾는 데 유용하다.

### 한계와 추가 확인 사항

연령이나 지역으로 대상을 제한할 때는 결측치 여부와 각 집단의 고객 수 차이를 함께 확인해야 한다. 단순 총액으로 비교한다면 고객 수가 많은 집단이 항상 크게 보일 수 있으므로 고객 1인당 평균 구매 금액도 추가로 확인할 필요가 있다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

orders.order_id 중복 수: 0


In [10]:
order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())


병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [11]:
order_sales = order_sales.drop(columns='_merge')

## 3) merge 검증
- 병합한 데이터: orders, order_items
- 사용한 key: order_id
- `validate` 결과: 'many_to_one' (order_items에 order_id는 여러 개일 수 있지만, orders에는 하나여야 하기에)
- `indicator` 결과: True
- 병합 전/후 행 수: 병합 전 행 수: 764
병합 후 행 수: 764

![merge 검증](images/step03_merge.png)

### 결과 관찰

order_items와 orders를 order_id 기준으로 병합했다. 병합 전후 행 수는 모두 764건으로 동일했다. order_items에는 한 주문에 여러 상품 행이 존재할 수 있고, orders에는 주문당 한 행만 존재해야 하므로 validate='many_to_one' 조건이 적절하다.
indicator=True로 생성한 _merge 결과 both가 764건, left_only와 right_only는 0건으로 확인되어 order_items의 각 주문상세가 orders의 주문 정보와 정상적으로 연결되었는지 검증했다.

### 나의 해석과 판단
이번 병합이 안전하다고 판단한 근거를 작성하세요. 행 수가 증가/감소했다면 이유를 설명하세요.

행 수가 유지되고 many_to_one 검증이 통과했다는 것은 orders의 중복 order_id로 인해 주문상세 행이 불필요하게 증가하지 않았다는 근거이다. 더불어 [병합 전 행 수 = 병합 후 행 수 = _merge 결과 both]인 것으로 보아 병합이 안전하게 이루어졌음을 확인할 수 있다.

### 업무·분석적 의미
잘못된 merge가 분석 결과에 어떤 영향을 줄 수 있는지 작성하세요.

잘못된 병합으로 주문 상세 행이 중복되면 line_total 합계, 카테고리별 판매 금액, 고객별 구매 금액이 모두 과대 계산될 수 있다. 반대로 기준 테이블이나 병합 방식을 잘못 선택해 행이 누락되면 금액이 과소 계산될 수 있다.

### 한계와 추가 확인 사항

위 검증은 orders와 order_items의 연결을 중심으로 수행했다. 이후 products, customers 병합에서도 각 키의 고유성, 미매칭 행, 병합 전후 행수를 동일하게 확인해야 한다. 

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [12]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())


날짜 변환 실패: 0


In [13]:
completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())

전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


In [14]:
completed_order_sales["order_status"].unique()

<ArrowStringArray>
['completed']
Length: 1, dtype: str

## 9. 상품 데이터 병합과 검증


In [15]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())


products.product_id 중복 수: 0


In [16]:
completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')

병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [17]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


In [18]:
product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))

,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


In [19]:
print(category_sales["total_sales"].sum())
print(completed_sales_items["line_total"].sum())

148990000
148990000


## 11. 월별 매출


In [20]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,11931000,14,852214.0
2,2025-11,12740000,15,849333.0
3,2025-12,20208000,22,918545.0
4,2026-01,17533000,19,922789.0
5,2026-02,4797000,7,685286.0
6,2026-03,19010000,25,760400.0
7,2026-04,10490000,16,655625.0
8,2026-05,12551000,16,784438.0
9,2026-06,16593000,19,873316.0


In [59]:
print("monthly_summary 합계: ", monthly_summary["total_sales"].sum())

monthly_summary 합계:  148990000


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [21]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


In [60]:
print("customer_sales 합계: ", customer_sales['total_sales'].sum())

customer_sales 합계:  148990000


## 4) completed 주문 범위와 집계
- 분석 범위 정의: order_sales['order_status'] == 'completed'
- 카테고리별 결과: 	

category	total_quantity	total_sales	sales_ratio

3	스포츠	295	31743000	21.31


- 상품별 결과:

product_id	product_name	category	total_quantity	total_sales

39	41	스포츠 상품 041	스포츠	35	5705000


- 월별 결과:

order_month	total_sales	order_count	average_order_value

3	2025-12	20208000	22	918545.0

11	2026-08	5103000	5	1020600.0


- 고객별 결과:

customer_label	city	order_count	total_sales

0	Customer 117	성남	5	4100000

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
수치에서 직접 확인한 상위/하위 또는 변화 패턴을 작성하세요.

카테고리별로는 스포츠가 31,743,000원으로 가장 높았고 전체의 21.31%를 차지했다. 상품별로는 스포츠 상품 041이 5,705,000원으로 가장 높은 판매 금액을 기록했다. 월별 결과에서 2025년 12월이 20,208,000원으로 total_sales는 가장 높지만, 2026년 8월이 average_order_value가 1,020,600원으로 가장 높다. 고객별로는 Customer 117이 4,100,000원, 5건의 주문으로 가장 높은 구매 금액을 기록했다.

### 나의 해석과 판단
어떤 결과가 가장 중요하다고 판단했는지와 이유를 작성하세요.

중요한 결과는 스포츠 카테고리의 높은 판매 금액과 2025년 12월의 판매 금액 최고치라고 생각한다. 스포츠는 전체 판매 금액의 약 5분의 1을 차지해 카테고리 차원에서의 영향력이 크다. 또한, 12월의 상승은 계절성·프로모션·주문 수 증가 등의 가능성을 추가로 확인할 필요가 있다.

### 업무·분석적 의미
다음 의사결정/분석으로 어떻게 연결할지 작성하세요.

스포츠 카테고리는 재고 확보, 인기 상품 관리, 추가 상품 추천의 우선 대상이 될 수 있다. 12월의 판매 증가 원인을 주문 수와 평균 주문 금액으로 나누어 확인하면 향후 프로모션 또는 재고 계획 수립에 활용할 수 있다. 고액 구매 고객은 재구매 유도나 고객 유지 전략의 후보군으로 볼 수 있다.

### 한계와 추가 확인 사항
`total_sales`가 회계상 순매출과 같다고 단정할 수 없는 이유를 포함하세요.

total_sales는 완료 주문의 quantity × unit_price 합계이다. 할인, 쿠폰, 배송비, 세금, 부분 환불 및 실제 환불 금액이 반영되지 않았으므로 회계상 순매출과 같다고 단정할 수 없다. 또한 2025년 9월과 2026년 6월은 월 전체 기간 데이터인지 확인해야 하며, 월별 비교 시 기간 길이가 다르면 왜곡될 수 있다.


In [ ]:
print(monthly_summary['total_sales'].sum() == completed_sales_items["line_total"].sum())
print(customer_sales['total_sales'].sum() == completed_sales_items["line_total"].sum())

True
True


## 5) 총합 일치 검증
- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계: 148990000
- 차이 여부: 없음

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
총합 검증이 필요한 이유와 불일치 시 확인할 순서를 작성하세요.

총합 검증은 서로 다른 기준으로 집계한 결과가 동일한 원본 완료 주문 데이터를 빠짐없이, 중복 없이 사용했는지 확인하기 위해 필요하다. 이번 분석에서는 원본 완료 주문의 line_total 합계와 카테고리·월별·고객별 합계가 모두 148,990,000원으로 일치했다. 따라서 각 집계 과정에서 금액이 누락되거나 중복 계산된 정황은 발견되지 않았다.

불일치가 발생한다면 다음 순서로 확인한다.
1. 완료 주문 필터가 모든 집계에 동일하게 적용되었는지 확인한다.
2. 병합 시 중복 키로 인해 행 수가 증가하지 않았는지 확인한다.
3. _merge 열을 통해 미매칭 행이 있는지 확인한다.
4. line_total의 결측치와 계산식을 확인한다.
5. groupby 대상 열의 결측치로 일부 행이 집계에서 제외되지 않았는지 확인한다.

## 13. 결과 저장하기


In [26]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


In [27]:
for path in sorted(REPORT_DIR.glob("ch04_*.csv")):
    print(
        path.name,
        path.exists(),
        path.stat().st_size,
    )

ch04_category_sales.csv True 254
ch04_customer_sales.csv True 3387
ch04_monthly_sales.csv True 443
ch04_product_sales.csv True 4409


In [28]:
check_category = pd.read_csv(
    REPORT_DIR / "ch04_category_sales.csv"
)

print(check_category.shape)
print(check_category.columns.tolist())

(7, 4)
['category', 'total_quantity', 'total_sales', 'sales_ratio']


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [29]:
category_sales = order_items.groupby('category')['line_total'].sum()

KeyError: 'category'

오류 이유: 현재 order_items에서 category를 기준으로 묶어, 카테고리별 매출을 확인하고자 한다. 그러나, category 칼럼은 products 파일에 존재하며, 기존의 order_items에는 line_total 칼럼이 존재하지 않는다.
따라서, (1) order_items에 unit price * quantity를 계산한 line_total 칼럼을 생성하고 (2) orders 파일과 order_id를 기준으로 병합하여 (3) order_status == completed인 주문만을 필터링한 후 (4) products 파일과 product_id를 기준으로 병합을 수행해야 한다.

In [40]:
# (0) order_id, product_id 고유성 확인
print(orders['order_id'].is_unique)
print(products['product_id'].is_unique)


True
True


In [41]:
# (1) line_total 칼럼 생성
order_items = order_items.copy()
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

display(order_items.head(10))

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000
5,6,3,33,5,124000,620000
6,7,3,20,3,80000,240000
7,8,3,11,5,115000,575000
8,9,4,45,3,62000,186000
9,10,4,84,4,78000,312000


In [42]:
# (2) orders와 order_items 병합
order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

# 여러 개의 order_items가 하나의 order_id에 매칭되므로 many_to_one 검증을 사용함
# how='left': orders에 있는 모든 order_id를 유지하고, order_items에 없는 경우 NaN으로 채움
# indicator=True: 병합 결과를 나타내는 _merge 컬럼을 생성함, orderr_items에만 있는 order_id 확인 가능

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())
order_sales = order_sales.drop(columns='_merge')

병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [43]:
# (3) order_status == 'completed'인 주문만 필터링
completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()
print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())

전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


In [45]:
# (4) completed_order_sales와 products 병합
completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

# 여러 개의 completed_order_sales가 하나의 product_id에 매칭되므로 many_to_one 검증을 사용함

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')

병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

In [ ]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
display(category_sales)

# completed_sales_items에 category 칼럼, quantity 칼럼, line_total 칼럼이 존재하므로 groupby와 agg를 사용하여 category별 총 판매 수량과 총 매출을 계산할 수 있음

,category,total_quantity,total_sales
3,스포츠,295,31743000
5,전자기기,259,26400000
2,생활용품,272,23915000
1,뷰티,223,23383000
4,식품,133,16573000
0,도서,149,16389000
6,패션,111,10587000


## LLM 입력 프롬프트

orders와 order_items를 사용해
completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.
- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기

In [46]:
# 1. 주문 ID가 orders 테이블에서 고유한지 확인
duplicate_order_ids = orders['order_id'].duplicated().sum()
print('orders의 중복 order_id 개수:', duplicate_order_ids)

assert duplicate_order_ids == 0, 'orders 테이블에 중복 order_id가 있습니다.'

orders의 중복 order_id 개수: 0


In [47]:
# 2. 주문일자 변환 및 변환 실패 확인
orders_clean = orders.copy()
orders_clean['order_date'] = pd.to_datetime(
    orders_clean['order_date'],
    errors='coerce',
)

date_parse_failures = orders_clean['order_date'].isna().sum()
print('주문일자 변환 실패 건수:', date_parse_failures)

assert date_parse_failures == 0, '변환할 수 없는 order_date 값이 있습니다.'

주문일자 변환 실패 건수: 0


In [48]:
# 3. 주문 상세별 금액 계산: 수량 × 단가
order_items_clean = order_items.copy()
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

In [49]:
# 4. orders와 order_items 병합
before_merge_rows = len(order_items_clean)

completed_order_sales = order_items_clean.merge(
    orders_clean[['order_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',   # 여러 order_items : 하나의 orders
    indicator=True,
)

after_merge_rows = len(completed_order_sales)

print('병합 전 행 수:', before_merge_rows)
print('병합 후 행 수:', after_merge_rows)

assert before_merge_rows == after_merge_rows, '병합 후 행 수가 달라졌습니다.'

병합 전 행 수: 764
병합 후 행 수: 764


In [50]:
# 5. 매칭되지 않은 주문 상세 확인
unmatched_items = completed_order_sales.loc[
    completed_order_sales['_merge'] != 'both'
]

print('미매칭 order_items 건수:', len(unmatched_items))
display(unmatched_items.head())

assert len(unmatched_items) == 0, 'orders에 없는 order_id가 order_items에 있습니다.'

미매칭 order_items 건수: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,order_date,order_status,_merge


In [51]:
# 6. completed 주문만 필터링하고 월 컬럼 생성
completed_order_sales = completed_order_sales.loc[
    completed_order_sales['order_status'].eq('completed')
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date']
    .dt.to_period('M')
    .astype(str)
)

In [52]:
# 7. 월별 완료 주문 금액과 고유 주문 수 계산
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

display(monthly_summary)

,order_month,total_sales,order_count
0,2025-09,2650000,5
1,2025-10,11931000,14
2,2025-11,12740000,15
3,2025-12,20208000,22
4,2026-01,17533000,19
5,2026-02,4797000,7
6,2026-03,19010000,25
7,2026-04,10490000,16
8,2026-05,12551000,16
9,2026-06,16593000,19


## 6) LLM pandas 코드 검증
- LLM Prompt 요약: orders, order_items, customers를 사용하여 완료 주문 기준 고객별 구매 금액을 계산하되, 병합 검증과 미매칭 확인을 포함한 Pandas 코드를 요청하였다.
- 제안 코드 요약: line_total을 생성하고, order_items와 orders를 many_to_one 방식으로 병합한 뒤 completed 주문만 필터링하였다. 이후 customers를 병합해 고객별 고유 주문 수와 구매 금액 합계를 계산하도록 제안했다.
- 실제 컬럼/범위와 맞지 않은 부분: category는 order_items가 아니라 products 테이블에 있으므로, 카테고리별 분석에는 products 병합이 추가로 필요했다. 또한 여러 번 indicator=True를 사용하면 이미 존재하는 _merge 열과 이름이 충돌할 수 있었다. 따라서, 해당 열을 drop하는 과정이 필요했다.
- 수정한 내용: 이전에 drop하지 않았다면, 상품 병합 시 indicator='_merge_products'처럼 indicator 열 이름을 구분했다. 각 병합에서 키의 고유성, 행 수 변화, 미매칭 여부를 확인했으며, 완료 주문만을 분석 범위로 명시했다.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM 코드가 실행되었다는 것만으로 분석이 맞다고 볼 수 없는 이유를 작성하세요.

LLM 코드가 오류 없이 실행되었다는 사실만으로 분석이 맞다고 볼 수는 없다. 존재하지 않는 컬럼을 사용하지 않았는지, 병합 관계가 실제 데이터 구조와 맞는지, 분석 범위가 질문과 일치하는지, 병합 후 중복이나 누락이 없는지를 별도로 검증해야 한다. 특히 데이터프레임 병합은 코드가 실행되더라도 금액이 중복되거나 일부 행이 누락될 수 있으므로 결과 총합 검증이 필요하다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [54]:
# 1. 40세 이상 고객 추출

customers_40_plus = customers[customers['age'] >= 40]
print(customers_40_plus.head(10))
print('40세 이상 고객 수:', len(customers_40_plus))

    customer_id name gender  age city signup_date
2             3  이경수      F   61   성남  2024-08-23
3             4  조영호      F   55   울산  2026-06-24
6             7  이상현      F   53   인천  2025-02-22
8             9  송지민      M   69   서울  2025-12-30
9            10  유도현      F   62   울산  2024-12-17
11           12  김정남      F   66   대전  2024-12-11
13           14  윤성호      F   40   대전  2025-04-16
14           15  장정식      M   69   서울  2026-08-15
15           16  강보람      M   52   부산  2024-10-03
17           18  김순옥      M   58   고양  2023-12-14
40세 이상 고객 수: 78


In [55]:
# 2. 상품 가격 낮은 순 10개 추출
display(products.sort_values('price', ascending=True).head(10))

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


In [56]:
# 3. 결제수단별 주문 수와 비율
payment_summary = (
    orders
    .groupby('payment_method', as_index=False)
    .agg(
        order_count=('order_id', 'count'),
    )
    .sort_values('order_count', ascending=False)
)

payment_summary['order_ratio'] = (
    payment_summary['order_count'] / payment_summary['order_count'].sum() * 100
).round(2)

display(payment_summary)

,payment_method,order_count,order_ratio
2,kakao_pay,79,26.33
3,naver_pay,77,25.67
0,bank_transfer,74,24.67
1,card,70,23.33


In [57]:
# 4. 카테고리별 평균 상품 가격
category_avg_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        avg_price=('price', 'mean'),
    )
    .sort_values('avg_price', ascending=False)
)
display(category_avg_price)

,category,avg_price
4,식품,137142.857143
1,뷰티,117687.500000
6,패션,115909.090909
3,스포츠,111578.947368
0,도서,106857.142857
5,전자기기,101588.235294
2,생활용품,96437.500000


In [58]:
# 5. 완료 주문과 전체 주문의 금액 차이
completed_sales_total = completed_order_sales['line_total'].sum()
all_sales_total = order_items['line_total'].sum()   
sales_difference = all_sales_total - completed_sales_total  
print(f"Sales difference: {sales_difference}")

Sales difference: 106620000


In [ ]:
# 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트 작성

orders, order_items, customers 테이블을 사용하여 completed 주문 기준 고객별 구매 금액을 분석하는 Pandas 코드를 작성해줘.

다음 조건을 반드시 포함해줘.

- order_items에서 line_total = quantity × unit_price를 계산할 것
- orders의 order_id 고유성을 확인할 것
- customers의 customer_id 고유성을 확인할 것
- order_items와 orders를 order_id 기준으로 left merge할 것
- 해당 병합에서 validate='many_to_one'과 indicator를 사용할 것
- 병합 전후 행 수를 확인할 것
- orders와 매칭되지 않은 order_items가 있는지 확인할 것
- order_status가 completed인 주문만 필터링할 것
- completed 주문 데이터와 customers를 customer_id 기준으로 left merge할 것
- 해당 병합에서도 validate와 indicator를 사용할 것
- customers와 매칭되지 않은 고객이 있는지 확인할 것
- 고객별 고유 주문 수(order_count)와 구매 금액 합계(total_sales)를 계산할 것
- total_sales 내림차순으로 정렬할 것
- customer_id, city, order_count, total_sales를 포함한 결과를 출력할 것
- 계산 결과는 완료 주문 기준 상품 금액 합계이며, 할인·배송비·세금·환불 정보가 없다면 회계상 순매출이라고 단정하지 않는다는 설명을 덧붙일 것
- 각 병합 검증 단계가 필요한 이유도 간단히 설명할 것

[Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: [예]

2. 병합 검증
- orders.order_id 중복: [orders.order_id 중복 수: 0]
- orders merge 전 행 수: [병합 전 행 수: 764]
- orders merge 후 행 수: [병합 후 행 수: 764]
- orders merge 미매칭: [both          764
left_only       0
right_only      0]
- products merge 전 행 수: [병합 전 행 수: 474]
- products merge 후 행 수: [병합 후 행 수: 474]
- products merge 미매칭: [both          474
left_only       0
right_only      0]

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: [날짜 변환 실패: 0]

4. 교차 검증
- 완료 주문 상세 line_total 합계: [완료 주문 매출: 148990000]
- category_sales 합계: [148990000]
- monthly_summary 합계: [monthly_summary 합계:  148990000]
- customer_sales 합계: [customer_sales 합계:  148990000]
- 합계 일치 여부: [PASS]

5. 저장 파일
- ch04_category_sales.csv: [존재]
- ch04_product_sales.csv: [존재]
- ch04_monthly_sales.csv: [존재]
- ch04_customer_sales.csv: [존재]

6. LLM 검증
- LLM이 빠뜨린 검증 항목: [completed 필터링 후 고유 order_id 수, line_total 결측치 및 비정상값 여부 확인]

7. 남은 질문
- [고객 정보가 없는 주문 상세 행이 있다면, 데이터 오류로 간주할 것인가, 별도 고객군으로 분석할 것인가]

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 7) Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 스포츠 카테고리가 완료 주문 기준 31,743,000원으로 가장 높은 판매 금액을 기록했고, 전체의 21.31%를 차지했다.
2. 2025년 12월의 판매 금액이 20,208,000원으로 가장 높았으며, 평균 주문 금액도 918,545원으로 높은 수준이었다. 그러나 가장 높은 평균 주문 금액은 가장 주문 수가 적은 2026년 8월이었다. 

### 그 결과를 뒷받침하는 수치/표

1. 스포츠: 판매 수량 295개, 판매 금액 31,743,000원, 판매 비중 21.31%
2. 2025년 12월: 판매 금액 20,208,000원, 주문 수 22건, 평균 주문 금액 918,545원
3. 전체 완료 주문 기준 상품 금액 합계: 148,990,000원

### 추가로 확인하고 싶은 질문

1. 2025년 12월 판매 증가가 주문 수 증가, 평균 주문 금액 상승, 특정 고가 상품 판매 중 무엇의 영향인지
2. 스포츠 카테고리의 높은 판매 금액은 여러 상품의 고른 판매 때문인지, 일부 인기 상품에 집중되어 있는지, 스포츠 카테고리 내 상품의 판매 비율은 각 어느정도인지
3. 고액 구매 고객이 반복 구매 고객인지, 특정 시점의 일회성 고가 구매 고객인지? 고액 구매 고객의 비율은 어느정도인지
4. 도시별 고객 수를 고려할 때, 고객 1인당 평균 구매 금액에 차이가 있는지

### 현재 결과의 한계

본 분석은 완료 주문의 상품별 수량과 단가를 바탕으로 한 금액 합계 분석이다. 환불 금액, 원가, 마진 등의 정보가 없으므로 순매출이나 수익성을 판단할 수 없다. 또한, 관찰된 차이의 원인을 인과적으로 설명할 수 없으므로, 계절성, 프로모션 등의 추가 데이터가 필요하다. 

## 최종 제출 체크
- [O] 핵심 셀 Output이 남아 있습니다.
- [O] merge와 총합 검증 Evidence가 있습니다.
- [O] 결과 관찰과 해석이 구분되어 있습니다.
- [O] LLM 코드를 검증했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.